# Exploració de dades

Catàleg, activitat, biaix de posició del sistema antic, i el vostre rendiment real
(`team_NN.feedback`). Guardeu aquí les gràfiques que vulgueu portar a la presentació.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
from src.rawtree_client import sql_df, TEAM
print("base de dades del vostre equip:", TEAM)  # ha de ser p.ex. 'team_07', mai 'team_NN'

import matplotlib.pyplot as plt
import pandas as pd

## Catàleg: mida, categories, preus

In [ ]:
catalog = sql_df("""
    SELECT product_id::String AS product_id, category::String AS category,
           subcategory::String AS subcategory, brand::String AS brand,
           price_eur::Float64 AS price_eur, is_active::Bool AS is_active
    FROM shop.catalog""")
print(len(catalog), "productes,", catalog.category.nunique(), "categories,",
      catalog.subcategory.nunique(), "subcategories,", catalog.brand.nunique(), "marques")
catalog.price_eur.describe()

In [ ]:
catalog.groupby("category").size().sort_values().plot.barh(title="Productes per categoria")
plt.tight_layout(); plt.savefig("productes_per_categoria.png"); plt.show()

## Activitat orgànica: view / add_to_cart / purchase

In [ ]:
events = sql_df("SELECT event_type::String AS event_type, count() AS n FROM shop.history_events GROUP BY event_type")
events.set_index("event_type").n.plot.bar(title="Esdeveniments per tipus (history_events)")
plt.tight_layout(); plt.savefig("esdeveniments_per_tipus.png"); plt.show()
events

## Biaix de posició del sistema antic (queries.sql #7)

Quants clics cau cada slot. Si el slot 1 concentra molts més clics que el 10,
l'ordre importa molt i val la pena posar-hi atenció extra en `recommend()`.

In [ ]:
slots = sql_df("""
    SELECT indexOf(shown, clicked) AS slot, count() AS clicks
    FROM shop.history_sessions WHERE clicked IS NOT NULL GROUP BY slot ORDER BY slot""")
slots.set_index("slot").clicks.plot.bar(title="Clics per slot (sistema antic)")
plt.tight_layout(); plt.savefig("biaix_posicio.png"); plt.show()
slots

## El vostre rendiment real (`TEAM.feedback`)

Queries #5 i #6 de `queries.sql`, adaptades: `team_NN` és un placeholder del README,
no un nom de taula real, així que fem servir `TEAM` (llegit de `.env`) en comptes d'escriure'l a pèl.

In [ ]:
perf = sql_df(f"""
    SELECT toStartOfHour(ts::DateTime64(3)) AS hour, served_by::String AS served_by,
           count() AS requests,
           100 * countIf(clicked != '') / count()   AS ctr_pct,
           100 * countIf(purchased != '') / count() AS conversion_pct,
           sum(revenue_eur::Float64) / count()      AS revenue_per_request
    FROM {TEAM}.feedback WHERE request_id != '__init__'
    GROUP BY hour, served_by ORDER BY hour DESC, served_by""")
perf.head(20)

In [ ]:
pivot = perf.pivot_table(index="hour", columns="served_by", values="conversion_pct")
pivot.plot(title="Conversió (%) per hora: team vs fallback")
plt.tight_layout(); plt.savefig("conversio_per_hora.png"); plt.show()

In [ ]:
reliability = sql_df(f"""
    SELECT fallback_reason::String AS fallback_reason, count() AS requests,
           quantile(0.5)(latency_ms::Float64)  AS p50_ms,
           quantile(0.95)(latency_ms::Float64) AS p95_ms
    FROM {TEAM}.feedback
    WHERE ts > now() - INTERVAL 30 MINUTE AND request_id != '__init__'
    GROUP BY fallback_reason""")
print("'' = contestat pel vostre sistema (served_by='team'); la resta són motius de fallback")
reliability